1 — Import Libraries and Configure Paths

In [ ]:
# ============================================================
# CELL 1 - IMPORT LIBRARIES AND CONFIGURE PATHS
# Purpose:
# Initialize the exploratory data analysis environment.
# ============================================================

from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick


# ------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "data").is_dir(), (
    "Project root was not found."
)


# ------------------------------------------------------------
# Input dataset
# ------------------------------------------------------------

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
    / "swift_payments_cleaned.csv"
)

assert DATA_PATH.is_file(), (
    f"Cleaned dataset not found: {DATA_PATH}"
)


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
CHART_DIR = PROJECT_ROOT / "outputs" / "charts"
RULE_DIR = PROJECT_ROOT / "outputs" / "rules"

for directory in [TABLE_DIR, CHART_DIR, RULE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Processing configuration
# ------------------------------------------------------------

CHUNK_SIZE = 25_000
MIN_GROUP_SIZE = 100

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["font.size"] = 11


# ------------------------------------------------------------
# Display setup
# ------------------------------------------------------------

print("=" * 65)
print("SWIFT DATATHON - EXPLORATORY DATA ANALYSIS")
print("=" * 65)

print("Dataset:", DATA_PATH)
print("Dataset size (MB):", round(
    DATA_PATH.stat().st_size / (1024 ** 2), 2
))
print("Chunk size:", CHUNK_SIZE)

print("\nEDA environment initialized successfully.")

2 — Validate Cleaned Dataset Schema

In [ ]:
# ============================================================
# CELL 2 - VALIDATE DATASET SCHEMA
# Purpose:
# Confirm that all required analytical fields are available.
# ============================================================

REQUIRED_COLUMNS = [
    "EPOCH_SECONDS",
    "CURRENCY_CODE",
    "USD_AMOUNT",
    "ORDERING_BIC8",
    "BENEFICIARY_BIC8",
    "BIC8_PATH",
    "TOTAL_DURATION",
    "CONTINUITY_QFLAG",
    "CURRENCY_CONVERSION_COUNT",
    "PARTICIPANT_COUNT",
    "TRANSACTION_TYPE_CODE",
    "ORDERING_TYPE",
    "BENEFICIARY_TYPE",
    "ANOMALY_TYPE",
    "CURRENCY_CODE_STD",
    "TRANSACTION_TYPE_CODE_STD",
    "CONTINUITY_QFLAG_STD",
    "PARTICIPANT_COUNT_NUM",
    "FX_COUNT_NUM",
    "TOTAL_DURATION_NUM",
    "USD_AMOUNT_NUM",
    "PARTY_PAIR",
    "HAS_PATH_GAP",
    "OBSERVED_BIC_COUNT",
    "HAS_ANOMALY_LABEL"
]

available_columns = pd.read_csv(
    DATA_PATH,
    nrows=0
).columns.tolist()

missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in available_columns
]

print("Available columns:", len(available_columns))
print("Required columns:", len(REQUIRED_COLUMNS))
print("Missing columns:", missing_columns)

assert not missing_columns, (
    "Required analytical columns are missing."
)

print("\nDataset schema validation passed.")

3 — Define EDA Metrics and Helper Functions

In [ ]:
# ============================================================
# CELL 3 - DEFINE EDA HELPER FUNCTIONS
# Purpose:
# Standardize calculations and chunk-based reading.
# ============================================================

def iter_cleaned_chunks(usecols=None):

    reader = pd.read_csv(
        DATA_PATH,
        usecols=usecols,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:
        yield chunk


def safe_percentage(numerator, denominator):

    if denominator == 0:
        return np.nan

    return numerator / denominator * 100


def safe_ratio(numerator, denominator):

    if denominator == 0:
        return np.nan

    return numerator / denominator


def create_frequency_table(counter, group_name):

    result = pd.DataFrame(
        counter.items(),
        columns=[group_name, "TRANSACTION_COUNT"]
    )

    total = result["TRANSACTION_COUNT"].sum()

    result["PERCENTAGE"] = (
        result["TRANSACTION_COUNT"] / total * 100
    )

    return result.sort_values(
        "TRANSACTION_COUNT",
        ascending=False
    ).reset_index(drop=True)


def export_table(df, filename):

    output_path = TABLE_DIR / filename

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Table exported:", output_path)


def export_chart(filename):

    output_path = CHART_DIR / filename

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    print("Chart exported:", output_path)


print("EDA helper functions initialized.")

4 — Dataset Overview & Main Category Distributions

In [ ]:
# ============================================================
# CELL 4 - DATASET OVERVIEW
# Purpose:
# Establish the overall payment population and distributions.
# ============================================================

total_records = 0

routing_counter = Counter()
currency_counter = Counter()
party_counter = Counter()
continuity_counter = Counter()
anomaly_counter = Counter()

overview_columns = [
    "TRANSACTION_TYPE_CODE_STD",
    "CURRENCY_CODE_STD",
    "PARTY_PAIR",
    "CONTINUITY_QFLAG_STD",
    "ANOMALY_TYPE_STD"
]

for chunk in iter_cleaned_chunks(overview_columns):

    total_records += len(chunk)

    for col, counter in [
        ("TRANSACTION_TYPE_CODE_STD", routing_counter),
        ("CURRENCY_CODE_STD", currency_counter),
        ("PARTY_PAIR", party_counter),
        ("CONTINUITY_QFLAG_STD", continuity_counter),
        ("ANOMALY_TYPE_STD", anomaly_counter)
    ]:

        values = (
            chunk[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        if col == "ANOMALY_TYPE_STD":
            values = values.replace("", "UNLABELED")
        else:
            values = values.replace("", "MISSING")

        counter.update(values.value_counts().to_dict())


# ------------------------------------------------------------
# Create frequency tables
# ------------------------------------------------------------

routing_table = create_frequency_table(
    routing_counter,
    "ROUTING_TYPE"
)

currency_table = create_frequency_table(
    currency_counter,
    "CURRENCY"
)

party_table = create_frequency_table(
    party_counter,
    "PARTY_PAIR"
)

continuity_table = create_frequency_table(
    continuity_counter,
    "CONTINUITY_FLAG"
)

anomaly_table = create_frequency_table(
    anomaly_counter,
    "ANOMALY_TYPE"
)


# ------------------------------------------------------------
# Export evidence tables
# ------------------------------------------------------------

export_table(routing_table, "eda_routing_distribution.csv")
export_table(currency_table, "eda_currency_distribution.csv")
export_table(party_table, "eda_party_distribution.csv")
export_table(continuity_table, "eda_continuity_distribution.csv")
export_table(anomaly_table, "eda_anomaly_distribution.csv")


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 65)
print("DATASET OVERVIEW")
print("=" * 65)

print("Total transactions:", f"{total_records:,}")

print("\nRouting Type Distribution:")
display(routing_table)

print("\nParty Pair Distribution:")
display(party_table)

print("\nContinuity Distribution:")
display(continuity_table)

print("\nTop 15 Currencies:")
display(currency_table.head(15))

print("\nAnomaly Distribution:")
display(anomaly_table)

5 — Visualize Routing Type Distribution

In [ ]:
# ============================================================
# CELL 5 - ROUTING TYPE DISTRIBUTION
# Purpose:
# Visualize the baseline composition of payment routing types.
# ============================================================

plot_data = routing_table.copy()

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    plot_data["ROUTING_TYPE"],
    plot_data["TRANSACTION_COUNT"]
)

ax.set_title(
    "Payment Routing Composition",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Routing Type")
ax.set_ylabel("Number of Transactions")

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

for bar, percentage in zip(
    bars,
    plot_data["PERCENTAGE"]
):
    ax.annotate(
        f"{percentage:.1f}%",
        (
            bar.get_x() + bar.get_width() / 2,
            bar.get_height()
        ),
        ha="center",
        va="bottom"
    )

plt.tight_layout()

export_chart("eda_routing_distribution.png")

plt.show()

6 — Routing Complexity Distribution

In [ ]:
# ============================================================
# CELL 6 - ROUTING COMPLEXITY DISTRIBUTION
# Purpose:
# Understand participant and FX conversion distributions.
# ============================================================

participant_counter = Counter()
fx_counter = Counter()

for chunk in iter_cleaned_chunks([
    "PARTICIPANT_COUNT_NUM",
    "FX_COUNT_NUM"
]):

    participants = pd.to_numeric(
        chunk["PARTICIPANT_COUNT_NUM"],
        errors="coerce"
    )

    fx_steps = pd.to_numeric(
        chunk["FX_COUNT_NUM"],
        errors="coerce"
    )

    participant_counter.update(
        participants.value_counts(dropna=False).to_dict()
    )

    fx_counter.update(
        fx_steps.value_counts(dropna=False).to_dict()
    )


participant_table = create_frequency_table(
    participant_counter,
    "PARTICIPANT_COUNT"
)

fx_table = create_frequency_table(
    fx_counter,
    "FX_COUNT"
)

export_table(
    participant_table,
    "eda_participant_distribution.csv"
)

export_table(
    fx_table,
    "eda_fx_distribution.csv"
)

print("Participant Count Distribution:")
display(participant_table)

print("\nFX Conversion Count Distribution:")
display(fx_table)

7 — Routing Type vs Participant Count

In [ ]:
# ============================================================
# CELL 7 - ROUTING TYPE VS PARTICIPANT COUNT
# Purpose:
# Explore whether routing types differ in participant count.
# ============================================================

routing_participant_counter = Counter()

for chunk in iter_cleaned_chunks([
    "TRANSACTION_TYPE_CODE_STD",
    "PARTICIPANT_COUNT_NUM"
]):

    grouped = (
        chunk.groupby(
            [
                "TRANSACTION_TYPE_CODE_STD",
                "PARTICIPANT_COUNT_NUM"
            ],
            dropna=False
        )
        .size()
    )

    routing_participant_counter.update(
        grouped.to_dict()
    )


rows = [
    {
        "ROUTING_TYPE": routing,
        "PARTICIPANT_COUNT": participants,
        "TRANSACTION_COUNT": count
    }
    for (routing, participants), count
    in routing_participant_counter.items()
]

routing_participant_table = pd.DataFrame(rows)

routing_participant_table["WITHIN_ROUTING_PERCENT"] = (
    routing_participant_table["TRANSACTION_COUNT"]
    /
    routing_participant_table.groupby(
        "ROUTING_TYPE"
    )["TRANSACTION_COUNT"].transform("sum")
    * 100
)

routing_participant_table = (
    routing_participant_table.sort_values(
        ["ROUTING_TYPE", "PARTICIPANT_COUNT"]
    )
)

export_table(
    routing_participant_table,
    "eda_routing_vs_participants.csv"
)

display(routing_participant_table)

8 — Routing Type vs FX Conversion Count

In [ ]:
# ============================================================
# CELL 8 - ROUTING TYPE VS FX COUNT
# Purpose:
# Investigate FX conversion patterns across routing categories.
# ============================================================

routing_fx_counter = Counter()

for chunk in iter_cleaned_chunks([
    "TRANSACTION_TYPE_CODE_STD",
    "FX_COUNT_NUM"
]):

    grouped = (
        chunk.groupby(
            [
                "TRANSACTION_TYPE_CODE_STD",
                "FX_COUNT_NUM"
            ],
            dropna=False
        )
        .size()
    )

    routing_fx_counter.update(grouped.to_dict())


routing_fx_table = pd.DataFrame([
    {
        "ROUTING_TYPE": routing,
        "FX_COUNT": fx,
        "TRANSACTION_COUNT": count
    }
    for (routing, fx), count
    in routing_fx_counter.items()
])

routing_fx_table["WITHIN_ROUTING_PERCENT"] = (
    routing_fx_table["TRANSACTION_COUNT"]
    /
    routing_fx_table.groupby(
        "ROUTING_TYPE"
    )["TRANSACTION_COUNT"].transform("sum")
    * 100
)

routing_fx_table = routing_fx_table.sort_values(
    ["ROUTING_TYPE", "FX_COUNT"]
)

export_table(
    routing_fx_table,
    "eda_routing_vs_fx.csv"
)

display(routing_fx_table)

9 — Payment Duration Distribution

In [ ]:
# ============================================================
# CELL 9 - PAYMENT DURATION DISTRIBUTION
# Purpose:
# Calculate exact payment duration statistics and percentiles.
# ============================================================

duration_parts = []

for chunk in iter_cleaned_chunks(["TOTAL_DURATION_NUM"]):

    values = pd.to_numeric(
        chunk["TOTAL_DURATION_NUM"],
        errors="coerce"
    )

    duration_parts.append(
        values.dropna().to_numpy(dtype=np.float64)
    )

all_durations = np.concatenate(duration_parts)

duration_stats = pd.DataFrame([
    {
        "METRIC": "Valid transactions",
        "VALUE": len(all_durations)
    },
    {
        "METRIC": "Mean duration (seconds)",
        "VALUE": np.mean(all_durations)
    },
    {
        "METRIC": "Median duration (seconds)",
        "VALUE": np.median(all_durations)
    },
    {
        "METRIC": "Minimum duration (seconds)",
        "VALUE": np.min(all_durations)
    },
    {
        "METRIC": "P90 duration (seconds)",
        "VALUE": np.percentile(all_durations, 90)
    },
    {
        "METRIC": "P95 duration (seconds)",
        "VALUE": np.percentile(all_durations, 95)
    },
    {
        "METRIC": "P99 duration (seconds)",
        "VALUE": np.percentile(all_durations, 99)
    },
    {
        "METRIC": "Maximum duration (seconds)",
        "VALUE": np.max(all_durations)
    }
])

export_table(
    duration_stats,
    "eda_duration_statistics.csv"
)

display(duration_stats)

print("Duration distribution analysis completed.")

10 — Visualize Payment Duration Distribution

In [ ]:
# ============================================================
# CELL 10 - IMPROVED PAYMENT DURATION VISUALIZATION
# Purpose:
# Examine short-duration concentration and the full
# payment duration distribution without modifying data.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ------------------------------------------------------------
# 1. Calculate duration statistics
# ------------------------------------------------------------

median_duration = np.median(all_durations)
p90_duration = np.percentile(all_durations, 90)
p95_duration = np.percentile(all_durations, 95)
p99_duration = np.percentile(all_durations, 99)

print("Median Duration:", round(median_duration, 2))
print("P90 Duration:", round(p90_duration, 2))
print("P95 Duration:", round(p95_duration, 2))
print("P99 Duration:", round(p99_duration, 2))


# ------------------------------------------------------------
# 2. Zoom into short-duration transactions
# ------------------------------------------------------------

short_durations = all_durations[
    all_durations <= 300
]

fig, ax = plt.subplots(figsize=(11, 6))

ax.hist(
    short_durations,
    bins=np.arange(0, 305, 5),
    edgecolor="white",
    alpha=0.85
)

ax.axvline(
    median_duration,
    linestyle="--",
    linewidth=2,
    label=f"Overall Median: {median_duration:.1f}s"
)

ax.set_title(
    "Payment Duration Distribution (0–300 Seconds)",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Payment Duration (Seconds)")
ax.set_ylabel("Transaction Count")

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

ax.legend()

plt.tight_layout()

export_chart("eda_duration_short_range.png")

plt.show()


# ------------------------------------------------------------
# 3. Visualize the full distribution using a log scale
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

ax.hist(
    all_durations,
    bins=np.logspace(
        np.log10(max(1, all_durations.min())),
        np.log10(all_durations.max()),
        80
    ),
    edgecolor="white",
    alpha=0.85
)

ax.set_xscale("log")

ax.axvline(
    median_duration,
    linestyle="--",
    linewidth=2,
    label=f"Median: {median_duration:.1f}s"
)

ax.set_title(
    "Full Payment Duration Distribution (Logarithmic Scale)",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Payment Duration (Seconds, Log Scale)")
ax.set_ylabel("Transaction Count")

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

ax.legend()

plt.tight_layout()

export_chart("eda_duration_log_distribution.png")

plt.show()


print("\nDuration visualizations completed.")
print("No transaction records were removed or modified.")

11 — Median Duration by Routing Type

In [ ]:
# ============================================================
# CELL 11 - DURATION BY ROUTING TYPE
# Purpose:
# Compare payment duration distributions across routing types.
# ============================================================

duration_by_route = defaultdict(list)

for chunk in iter_cleaned_chunks([
    "TRANSACTION_TYPE_CODE_STD",
    "TOTAL_DURATION_NUM"
]):

    chunk["TOTAL_DURATION_NUM"] = pd.to_numeric(
        chunk["TOTAL_DURATION_NUM"],
        errors="coerce"
    )

    for routing, group in chunk.groupby(
        "TRANSACTION_TYPE_CODE_STD",
        dropna=False
    ):

        key = (
            str(routing)
            if pd.notna(routing)
            else "MISSING"
        )

        values = (
            group["TOTAL_DURATION_NUM"]
            .dropna()
            .to_numpy(dtype=np.float64)
        )

        duration_by_route[key].append(values)


route_duration_rows = []

for routing, parts in duration_by_route.items():

    values = np.concatenate(parts)

    if len(values) == 0:
        continue

    route_duration_rows.append({
        "ROUTING_TYPE": routing,
        "VALID_DURATION_COUNT": len(values),
        "MEAN_DURATION": np.mean(values),
        "MEDIAN_DURATION": np.median(values),
        "P90_DURATION": np.percentile(values, 90),
        "P95_DURATION": np.percentile(values, 95),
        "MIN_DURATION": np.min(values),
        "MAX_DURATION": np.max(values)
    })


route_duration_table = pd.DataFrame(
    route_duration_rows
).sort_values("MEDIAN_DURATION")

export_table(
    route_duration_table,
    "eda_duration_by_routing.csv"
)

display(route_duration_table)

12 — Visualize Median Duration by Routing Type

In [ ]:
# ============================================================
# CELL 12 - VISUALIZE DURATION BY ROUTING TYPE
# Purpose:
# Compare median durations across routing categories.
# ============================================================

plot_data = route_duration_table.copy()

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    plot_data["ROUTING_TYPE"],
    plot_data["MEDIAN_DURATION"]
)

ax.set_title(
    "Median Payment Duration by Routing Type",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Routing Type")
ax.set_ylabel("Median Duration (Seconds)")

max_value = plot_data["MEDIAN_DURATION"].max()

ax.set_ylim(0, max_value * 1.18 if max_value > 0 else 1)

for bar in bars:

    height = bar.get_height()

    ax.annotate(
        f"{height:,.1f}s",
        (
            bar.get_x() + bar.get_width() / 2,
            height
        ),
        ha="center",
        va="bottom"
    )

plt.tight_layout()

export_chart("eda_median_duration_by_routing.png")

plt.show()

13 — Participant Count vs Payment Duration

In [ ]:
# ============================================================
# CELL 13 - PARTICIPANT COUNT VS DURATION
# Purpose:
# Investigate how duration varies with routing complexity.
# ============================================================

duration_by_participant = defaultdict(list)

for chunk in iter_cleaned_chunks([
    "PARTICIPANT_COUNT_NUM",
    "TOTAL_DURATION_NUM"
]):

    chunk["PARTICIPANT_COUNT_NUM"] = pd.to_numeric(
        chunk["PARTICIPANT_COUNT_NUM"],
        errors="coerce"
    )

    chunk["TOTAL_DURATION_NUM"] = pd.to_numeric(
        chunk["TOTAL_DURATION_NUM"],
        errors="coerce"
    )

    valid_chunk = chunk.dropna(
        subset=[
            "PARTICIPANT_COUNT_NUM",
            "TOTAL_DURATION_NUM"
        ]
    )

    for count, group in valid_chunk.groupby(
        "PARTICIPANT_COUNT_NUM"
    ):

        duration_by_participant[count].append(
            group["TOTAL_DURATION_NUM"].to_numpy(
                dtype=np.float64
            )
        )


participant_duration_rows = []

for count, parts in duration_by_participant.items():

    values = np.concatenate(parts)

    participant_duration_rows.append({
        "PARTICIPANT_COUNT": count,
        "TRANSACTION_COUNT": len(values),
        "MEDIAN_DURATION": np.median(values),
        "MEAN_DURATION": np.mean(values),
        "P90_DURATION": np.percentile(values, 90)
    })


participant_duration_table = pd.DataFrame(
    participant_duration_rows
).sort_values("PARTICIPANT_COUNT")

export_table(
    participant_duration_table,
    "eda_duration_by_participants.csv"
)

display(participant_duration_table)


# ------------------------------------------------------------
# Visualize the relationship
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    participant_duration_table["PARTICIPANT_COUNT"],
    participant_duration_table["MEDIAN_DURATION"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Median Payment Duration by Participant Count",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Number of Participants")
ax.set_ylabel("Median Duration (Seconds)")

ax.grid(alpha=0.25)

plt.tight_layout()

export_chart("eda_duration_by_participants.png")

plt.show()

14 — FX Conversion Count vs Payment Duration

In [ ]:
# ============================================================
# CELL 14 - FX CONVERSION VS DURATION
# Purpose:
# Explore payment duration patterns across FX conversion counts.
# ============================================================

duration_by_fx = defaultdict(list)

for chunk in iter_cleaned_chunks([
    "FX_COUNT_NUM",
    "TOTAL_DURATION_NUM"
]):

    chunk["FX_COUNT_NUM"] = pd.to_numeric(
        chunk["FX_COUNT_NUM"],
        errors="coerce"
    )

    chunk["TOTAL_DURATION_NUM"] = pd.to_numeric(
        chunk["TOTAL_DURATION_NUM"],
        errors="coerce"
    )

    valid_chunk = chunk.dropna(
        subset=[
            "FX_COUNT_NUM",
            "TOTAL_DURATION_NUM"
        ]
    )

    for fx_count, group in valid_chunk.groupby("FX_COUNT_NUM"):

        duration_by_fx[fx_count].append(
            group["TOTAL_DURATION_NUM"].to_numpy(
                dtype=np.float64
            )
        )


fx_duration_rows = []

for fx_count, parts in duration_by_fx.items():

    values = np.concatenate(parts)

    fx_duration_rows.append({
        "FX_COUNT": fx_count,
        "TRANSACTION_COUNT": len(values),
        "MEDIAN_DURATION": np.median(values),
        "MEAN_DURATION": np.mean(values),
        "P90_DURATION": np.percentile(values, 90)
    })


fx_duration_table = pd.DataFrame(
    fx_duration_rows
).sort_values("FX_COUNT")

export_table(
    fx_duration_table,
    "eda_duration_by_fx.csv"
)

display(fx_duration_table)


# ------------------------------------------------------------
# Visualize FX count and duration
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    fx_duration_table["FX_COUNT"],
    fx_duration_table["MEDIAN_DURATION"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Median Payment Duration by FX Conversion Count",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Number of FX Conversions")
ax.set_ylabel("Median Duration (Seconds)")

ax.grid(alpha=0.25)

plt.tight_layout()

export_chart("eda_duration_by_fx.png")

plt.show()

15 — Routing Path Continuity Analysis

In [ ]:
# ============================================================
# CELL 15 - ROUTING PATH CONTINUITY ANALYSIS
# Purpose:
# Compare routing gap rates across transaction types.
# ============================================================

continuity_counter = Counter()

for chunk in iter_cleaned_chunks([
    "TRANSACTION_TYPE_CODE_STD",
    "HAS_PATH_GAP"
]):

    grouped = (
        chunk.groupby(
            [
                "TRANSACTION_TYPE_CODE_STD",
                "HAS_PATH_GAP"
            ],
            dropna=False
        )
        .size()
    )

    continuity_counter.update(grouped.to_dict())


continuity_rows = [
    {
        "ROUTING_TYPE": routing,
        "HAS_PATH_GAP": gap,
        "TRANSACTION_COUNT": count
    }
    for (routing, gap), count
    in continuity_counter.items()
]

routing_continuity_table = pd.DataFrame(
    continuity_rows
)

routing_continuity_table["ROUTING_TOTAL"] = (
    routing_continuity_table.groupby(
        "ROUTING_TYPE"
    )["TRANSACTION_COUNT"].transform("sum")
)

routing_continuity_table["PERCENT_WITHIN_ROUTE"] = (
    routing_continuity_table["TRANSACTION_COUNT"]
    / routing_continuity_table["ROUTING_TOTAL"]
    * 100
)

export_table(
    routing_continuity_table,
    "eda_routing_continuity.csv"
)

display(routing_continuity_table)


# ------------------------------------------------------------
# Calculate routing gap rates
# ------------------------------------------------------------

gap_rate_table = (
    routing_continuity_table[
        routing_continuity_table["HAS_PATH_GAP"] == "Y"
    ][
        [
            "ROUTING_TYPE",
            "TRANSACTION_COUNT",
            "ROUTING_TOTAL",
            "PERCENT_WITHIN_ROUTE"
        ]
    ]
    .copy()
)

gap_rate_table = gap_rate_table.rename(columns={
    "TRANSACTION_COUNT": "GAP_COUNT",
    "PERCENT_WITHIN_ROUTE": "GAP_RATE_PERCENT"
})

export_table(
    gap_rate_table,
    "eda_gap_rate_by_routing.csv"
)

display(gap_rate_table)


# ------------------------------------------------------------
# Visualize gap rates
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    gap_rate_table["ROUTING_TYPE"],
    gap_rate_table["GAP_RATE_PERCENT"]
)

ax.set_title(
    "Routing Gap Rate by Transaction Type",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Routing Type")
ax.set_ylabel("Gap Rate (%)")

ax.set_ylim(
    0,
    max(1, gap_rate_table["GAP_RATE_PERCENT"].max() * 1.2)
)

for bar in bars:
    ax.annotate(
        f"{bar.get_height():.2f}%",
        (
            bar.get_x() + bar.get_width() / 2,
            bar.get_height()
        ),
        ha="center",
        va="bottom"
    )

plt.tight_layout()

export_chart("eda_gap_rate_by_routing.png")

plt.show()

16 — Currency and Party Relationship Analysis

In [ ]:
# ============================================================
# CELL 16 - CURRENCY AND PARTY RELATIONSHIP ANALYSIS
# Purpose:
# Compare currency preferences across party relationship types.
# ============================================================

currency_party_counter = Counter()

for chunk in iter_cleaned_chunks([
    "PARTY_PAIR",
    "CURRENCY_CODE_STD"
]):

    grouped = (
        chunk.groupby(
            ["PARTY_PAIR", "CURRENCY_CODE_STD"],
            dropna=False
        )
        .size()
    )

    currency_party_counter.update(grouped.to_dict())


currency_party_rows = [
    {
        "PARTY_PAIR": party,
        "CURRENCY": currency,
        "TRANSACTION_COUNT": count
    }
    for (party, currency), count
    in currency_party_counter.items()
]

currency_party_table = pd.DataFrame(
    currency_party_rows
)

currency_party_table["PARTY_TOTAL"] = (
    currency_party_table.groupby(
        "PARTY_PAIR"
    )["TRANSACTION_COUNT"].transform("sum")
)

currency_party_table["WITHIN_PARTY_PERCENT"] = (
    currency_party_table["TRANSACTION_COUNT"]
    / currency_party_table["PARTY_TOTAL"]
    * 100
)

currency_party_table = currency_party_table.sort_values(
    ["PARTY_PAIR", "TRANSACTION_COUNT"],
    ascending=[True, False]
)

export_table(
    currency_party_table,
    "eda_currency_by_party.csv"
)

print("Top currencies within each party relationship:")

display(
    currency_party_table.groupby(
        "PARTY_PAIR",
        group_keys=False
    ).head(10)
)

17 — Anomaly Distribution and Preliminary Screening

In [ ]:
# ============================================================
# CELL 17 - ANOMALY DISTRIBUTION AND SCREENING
# Purpose:
# Examine labeled anomaly populations and sample sizes.
# ============================================================

anomaly_screening = anomaly_table.copy()

anomaly_screening["SAMPLE_SIZE_STATUS"] = np.where(
    anomaly_screening["TRANSACTION_COUNT"] >= 100,
    "SUFFICIENT_FOR_INITIAL_EXPLORATION",
    "SMALL_SAMPLE_REQUIRES_CAUTION"
)

anomaly_screening.loc[
    anomaly_screening["ANOMALY_TYPE"] == "UNLABELED",
    "SAMPLE_SIZE_STATUS"
] = "UNLABELED_POPULATION"

export_table(
    anomaly_screening,
    "eda_anomaly_screening.csv"
)

display(anomaly_screening)

print(
    "Note: Sample-size screening is descriptive only. "
    "It does not establish anomaly-generating rules."
)

18 — Token Pattern Overview

In [ ]:
# ============================================================
# CELL 18 - TOKEN PATTERN OVERVIEW
# Purpose:
# Inspect basic structural properties of engineered token data.
# ============================================================

token_stats = {
    "RECORDS": 0,
    "ORDERING_BLANK": 0,
    "BENEFICIARY_BLANK": 0,
    "ORDERING_LENGTH_SUM": 0,
    "BENEFICIARY_LENGTH_SUM": 0
}

for chunk in iter_cleaned_chunks([
    "ORDERING_TOKENS",
    "BENEFICIARY_TOKENS"
]):

    ordering = (
        chunk["ORDERING_TOKENS"]
        .fillna("")
        .astype(str)
    )

    beneficiary = (
        chunk["BENEFICIARY_TOKENS"]
        .fillna("")
        .astype(str)
    )

    token_stats["RECORDS"] += len(chunk)

    token_stats["ORDERING_BLANK"] += int(
        ordering.str.strip().eq("").sum()
    )

    token_stats["BENEFICIARY_BLANK"] += int(
        beneficiary.str.strip().eq("").sum()
    )

    token_stats["ORDERING_LENGTH_SUM"] += int(
        ordering.str.len().sum()
    )

    token_stats["BENEFICIARY_LENGTH_SUM"] += int(
        beneficiary.str.len().sum()
    )


token_overview = pd.DataFrame([
    {
        "METRIC": "Ordering token blank records",
        "VALUE": token_stats["ORDERING_BLANK"]
    },
    {
        "METRIC": "Beneficiary token blank records",
        "VALUE": token_stats["BENEFICIARY_BLANK"]
    },
    {
        "METRIC": "Average ordering token string length",
        "VALUE": (
            token_stats["ORDERING_LENGTH_SUM"]
            / token_stats["RECORDS"]
        )
    },
    {
        "METRIC": "Average beneficiary token string length",
        "VALUE": (
            token_stats["BENEFICIARY_LENGTH_SUM"]
            / token_stats["RECORDS"]
        )
    }
])

export_table(
    token_overview,
    "eda_token_overview.csv"
)

display(token_overview)

19 — Generate Candidate Hypotheses Register

In [ ]:
# ============================================================
# CELL 19 - GENERATE CANDIDATE HYPOTHESES
# Purpose:
# Prepare a structured research register for rule discovery.
# ============================================================

hypotheses = [
    {
        "ID": "H01",
        "FEATURE_GROUP": "Routing",
        "HYPOTHESIS": (
            "Routing type is associated with participant count."
        ),
        "TARGET_NOTEBOOK": "03_bank_routing.ipynb"
    },
    {
        "ID": "H02",
        "FEATURE_GROUP": "Routing",
        "HYPOTHESIS": (
            "FX conversion count is associated with routing type."
        ),
        "TARGET_NOTEBOOK": "03_bank_routing.ipynb"
    },
    {
        "ID": "H03",
        "FEATURE_GROUP": "Routing",
        "HYPOTHESIS": (
            "Certain currencies have different routing distributions."
        ),
        "TARGET_NOTEBOOK": "03_bank_routing.ipynb"
    },
    {
        "ID": "H04",
        "FEATURE_GROUP": "Routing",
        "HYPOTHESIS": (
            "Routing gaps are concentrated in specific routing groups."
        ),
        "TARGET_NOTEBOOK": "03_bank_routing.ipynb"
    },
    {
        "ID": "H05",
        "FEATURE_GROUP": "Routing",
        "HYPOTHESIS": (
            "Specific BIC pairs show recurring route preferences."
        ),
        "TARGET_NOTEBOOK": "03_bank_routing.ipynb"
    },
    {
        "ID": "H06",
        "FEATURE_GROUP": "Duration",
        "HYPOTHESIS": (
            "Payment duration differs across routing types."
        ),
        "TARGET_NOTEBOOK": "04_payment_duration.ipynb"
    },
    {
        "ID": "H07",
        "FEATURE_GROUP": "Duration",
        "HYPOTHESIS": (
            "Payment duration is associated with participant count."
        ),
        "TARGET_NOTEBOOK": "04_payment_duration.ipynb"
    },
    {
        "ID": "H08",
        "FEATURE_GROUP": "Duration",
        "HYPOTHESIS": (
            "FX conversion count is associated with payment duration."
        ),
        "TARGET_NOTEBOOK": "04_payment_duration.ipynb"
    },
    {
        "ID": "H09",
        "FEATURE_GROUP": "Duration",
        "HYPOTHESIS": (
            "Routing type remains associated with duration "
            "after accounting for participant count."
        ),
        "TARGET_NOTEBOOK": "04_payment_duration.ipynb"
    },
    {
        "ID": "H10",
        "FEATURE_GROUP": "Duration",
        "HYPOTHESIS": (
            "Routing path continuity is associated with duration."
        ),
        "TARGET_NOTEBOOK": "04_payment_duration.ipynb"
    },
    {
        "ID": "H11",
        "FEATURE_GROUP": "Currency",
        "HYPOTHESIS": (
            "Currency selection differs by party relationship."
        ),
        "TARGET_NOTEBOOK": "05_currency_party.ipynb"
    },
    {
        "ID": "H12",
        "FEATURE_GROUP": "Currency",
        "HYPOTHESIS": (
            "Currency and party relationship interact "
            "with routing preferences."
        ),
        "TARGET_NOTEBOOK": "05_currency_party.ipynb"
    },
    {
        "ID": "H13",
        "FEATURE_GROUP": "Anomaly",
        "HYPOTHESIS": (
            "Duplicate payment labels correspond to repeated "
            "sender-beneficiary-amount patterns."
        ),
        "TARGET_NOTEBOOK": "06_anomaly_analysis.ipynb"
    },
    {
        "ID": "H14",
        "FEATURE_GROUP": "Anomaly",
        "HYPOTHESIS": (
            "Anomaly categories show distinct payment "
            "amount or routing patterns."
        ),
        "TARGET_NOTEBOOK": "06_anomaly_analysis.ipynb"
    },
    {
        "ID": "H15",
        "FEATURE_GROUP": "Tokens",
        "HYPOTHESIS": (
            "Engineered token patterns are associated "
            "with party type or anomaly labels."
        ),
        "TARGET_NOTEBOOK": "07_tokens.ipynb"
    }
]

hypothesis_register = pd.DataFrame(hypotheses)

hypothesis_register["STATUS"] = "NOT_TESTED"
hypothesis_register["EVIDENCE"] = ""
hypothesis_register["CONFIDENCE"] = ""
hypothesis_register["BUSINESS_IMPLICATION"] = ""

hypothesis_path = (
    RULE_DIR / "candidate_hypotheses.csv"
)

hypothesis_register.to_csv(
    hypothesis_path,
    index=False,
    encoding="utf-8-sig"
)

print("Candidate hypotheses:", len(hypothesis_register))
print("Output:", hypothesis_path)

display(hypothesis_register)

20 — Final EDA Validation & Output Summary

In [ ]:
# ============================================================
# CELL 20 - FINAL EDA VALIDATION
# Purpose:
# Verify record coverage and summarize the analysis outputs.
# ============================================================

EXPECTED_RECORDS = 1_048_575

assert total_records == EXPECTED_RECORDS, (
    "EDA record count does not match the expected dataset."
)

assert routing_table["TRANSACTION_COUNT"].sum() == total_records

assert currency_table["TRANSACTION_COUNT"].sum() == total_records

assert party_table["TRANSACTION_COUNT"].sum() == total_records

assert continuity_table["TRANSACTION_COUNT"].sum() == total_records

assert anomaly_table["TRANSACTION_COUNT"].sum() == total_records


# ------------------------------------------------------------
# Validate duration population
# ------------------------------------------------------------

duration_valid_count = len(all_durations)

print("=" * 65)
print("SWIFT DATATHON - EDA VALIDATION")
print("=" * 65)

print("Expected transactions:", f"{EXPECTED_RECORDS:,}")
print("EDA transactions:", f"{total_records:,}")

print("Valid duration records:", f"{duration_valid_count:,}")
print("Routing categories:", len(routing_table))
print("Currency categories:", len(currency_table))
print("Party relationship groups:", len(party_table))
print("Candidate hypotheses:", len(hypothesis_register))

print("\nEvidence Tables:")
print(TABLE_DIR)

print("\nCharts:")
print(CHART_DIR)

print("\nHypothesis Register:")
print(hypothesis_path)

print("\nEDA completed successfully.")
print("No original transaction records were modified.")